# Repair / Policy Loop Analysis Notebook

이 노트북은 `step_logs.jsonl`, `trajectory_logs.jsonl`, `summary.json`을 읽어 다음을 계산합니다.

- 전체 성능 요약
- 평균 calls / tokens
- problem-level repair 사용률
- problem-level repair 복구 성공률
- call-level repair 호출 수 및 성공률
- stage별 token 통계
- failure type / transition 분석

`RESULT_DIR`만 본인 실험 결과 폴더로 바꿔서 실행하면 됩니다.


In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# RESULT_DIR = Path("results/phase2_qwen/humaneval/repair")
# RESULT_DIR = Path("results/phase2_qwen/humaneval/rpe_policy_v2")
RESULT_DIR = Path("../../results/phase2_qwen/humaneval/repair")

STEP_LOG_PATH = RESULT_DIR / "step_logs.jsonl"
TRAJ_LOG_PATH = RESULT_DIR / "trajectory_logs.jsonl"
SUMMARY_PATH = RESULT_DIR / "summary.json"

print("RESULT_DIR:", RESULT_DIR)
print("step_logs exists:", STEP_LOG_PATH.exists())
print("trajectory_logs exists:", TRAJ_LOG_PATH.exists())
print("summary exists:", SUMMARY_PATH.exists())


RESULT_DIR: ../../results/phase2_qwen/humaneval/repair
step_logs exists: True
trajectory_logs exists: True
summary exists: True


In [2]:
def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

step_df = read_jsonl(STEP_LOG_PATH)
traj_df = read_jsonl(TRAJ_LOG_PATH)

summary = {}
if SUMMARY_PATH.exists():
    with SUMMARY_PATH.open("r", encoding="utf-8") as f:
        summary = json.load(f)

print("step_df:", step_df.shape)
print("traj_df:", traj_df.shape)

display(step_df.head())
display(traj_df.head())


step_df: (792, 29)
traj_df: (164, 17)


,run_id,dataset,problem_id,method,trajectory_id,step_id,call_index,candidate_id,stage,is_retry,...,status,error_type,error_stage,error_message,tests_passed,tests_total,code_length,selected,selection_rank,entry_point
0,phase2_qwen25coder7b_humaneval_repair_0424033048,humaneval,HumanEval/0,repair_loop,HumanEval/0_run0,0,0,0,generate,False,...,PASS,None,None,None,None,None,528,None,None,has_close_elements
1,phase2_qwen25coder7b_humaneval_repair_0424033048,humaneval,HumanEval/1,repair_loop,HumanEval/1_run0,0,0,0,generate,False,...,PASS,None,None,None,None,None,862,None,None,separate_paren_groups
2,phase2_qwen25coder7b_humaneval_repair_0424033048,humaneval,HumanEval/2,repair_loop,HumanEval/2_run0,0,0,0,generate,False,...,EXEC_FAIL:AssertionError,AssertionError,exec,"Traceback (most recent call last):\n File ""/h...",None,None,572,None,None,truncate_number
3,phase2_qwen25coder7b_humaneval_repair_0424033048,humaneval,HumanEval/2,repair_loop,HumanEval/2_run0,1,1,0,repair,False,...,EXEC_FAIL:NameError,NameError,exec,"Traceback (most recent call last):\n File ""/h...",None,None,815,None,None,truncate_number
4,phase2_qwen25coder7b_humaneval_repair_0424033048,humaneval,HumanEval/2,repair_loop,HumanEval/2_run0,2,2,0,repair,False,...,EXEC_FAIL:NameError,NameError,exec,"Traceback (most recent call last):\n File ""/h...",None,None,838,None,None,truncate_number


,run_id,dataset,problem_id,method,trajectory_id,num_steps,call_count,final_status,failure_family,final_tests_passed,final_tests_total,total_tokens,total_latency,num_exec_fail,num_test_fail,transition_path,budget_used
0,phase2_qwen25coder7b_humaneval_repair_0424033048,humaneval,HumanEval/0,repair_loop,HumanEval/0_run0,1,1,PASS,PASS,None,None,180,3.446053,0,0,[PASS],"{'tokens': 180, 'calls': 1, 'latency': 3.44605..."
1,phase2_qwen25coder7b_humaneval_repair_0424033048,humaneval,HumanEval/1,repair_loop,HumanEval/1_run0,1,1,PASS,PASS,None,None,225,1.600188,0,0,[PASS],"{'tokens': 225, 'calls': 1, 'latency': 1.60018..."
2,phase2_qwen25coder7b_humaneval_repair_0424033048,humaneval,HumanEval/2,repair_loop,HumanEval/2_run0,20,20,EXEC_FAIL:NameError,EXEC_FAIL,None,None,25077,132.072788,20,0,"[EXEC_FAIL:AssertionError, EXEC_FAIL:NameError...","{'tokens': 25077, 'calls': 20, 'latency': 132...."
3,phase2_qwen25coder7b_humaneval_repair_0424033048,humaneval,HumanEval/3,repair_loop,HumanEval/3_run0,1,1,PASS,PASS,None,None,165,0.762759,0,0,[PASS],"{'tokens': 165, 'calls': 1, 'latency': 0.76275..."
4,phase2_qwen25coder7b_humaneval_repair_0424033048,humaneval,HumanEval/4,repair_loop,HumanEval/4_run0,1,1,PASS,PASS,None,None,475,4.851071,0,0,[PASS],"{'tokens': 475, 'calls': 1, 'latency': 4.85107..."


## 1. Overall Performance Summary

In [3]:
def get_summary_value(summary, *keys, default=None):
    cur = summary
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

overall = {
    "total": get_summary_value(summary, "total_problems", default=len(traj_df)),
    "num_pass": get_summary_value(summary, "num_pass", default=(traj_df["final_status"].eq("PASS").sum() if "final_status" in traj_df else None)),
    "pass@1": get_summary_value(summary, "pass_at_1", default=None),
    "execution_success_rate": get_summary_value(summary, "execution_success_rate", default=None),
    "conditional_pass": get_summary_value(summary, "conditional_pass", default=None),
}

overall_df = pd.DataFrame([overall])
display(overall_df)


,total,num_pass,pass@1,execution_success_rate,conditional_pass
0,164,135,0.823171,0.865854,0.950704


## 2. Token / Call Statistics

`step_logs` 기준으로 stage별 호출 수와 token min/avg/max를 계산합니다.


In [4]:
token_cols = ["input_tokens", "output_tokens", "total_tokens"]
for c in token_cols:
    if c in step_df:
        step_df[c] = pd.to_numeric(step_df[c], errors="coerce")

stage_token_stats = (
    step_df
    .groupby(["dataset", "method", "stage"], dropna=False)
    .agg(
        num_calls=("stage", "size"),
        input_min=("input_tokens", "min"),
        input_avg=("input_tokens", "mean"),
        input_max=("input_tokens", "max"),
        output_min=("output_tokens", "min"),
        output_avg=("output_tokens", "mean"),
        output_max=("output_tokens", "max"),
        total_min=("total_tokens", "min"),
        total_avg=("total_tokens", "mean"),
        total_max=("total_tokens", "max"),
    )
    .round(2)
)

display(stage_token_stats)


num_calls  input_min  input_avg  input_max  \
dataset   method      stage                                                  
humaneval repair_loop generate        164         38     134.09        391   
                      repair          628        308     735.68       1350   

                                output_min  output_avg  output_max  total_min  \
dataset   method      stage                                                     
humaneval repair_loop generate          14      161.24         512         63   
                      repair            10      309.69         512        356   

                                total_avg  total_max  
dataset   method      stage                           
humaneval repair_loop generate     295.33        763  
                      repair      1045.37       1817

In [5]:
# Problem-level total cost
cost_summary = {}
if not traj_df.empty:
    for col in ["call_count", "total_tokens", "total_latency"]:
        if col in traj_df:
            traj_df[col] = pd.to_numeric(traj_df[col], errors="coerce")

    cost_summary = {
        "avg_calls_per_problem": traj_df["call_count"].mean() if "call_count" in traj_df else np.nan,
        "min_calls_per_problem": traj_df["call_count"].min() if "call_count" in traj_df else np.nan,
        "max_calls_per_problem": traj_df["call_count"].max() if "call_count" in traj_df else np.nan,
        "avg_tokens_per_problem": traj_df["total_tokens"].mean() if "total_tokens" in traj_df else np.nan,
        "min_tokens_per_problem": traj_df["total_tokens"].min() if "total_tokens" in traj_df else np.nan,
        "max_tokens_per_problem": traj_df["total_tokens"].max() if "total_tokens" in traj_df else np.nan,
    }

display(pd.DataFrame([cost_summary]).round(2))


,avg_calls_per_problem,min_calls_per_problem,max_calls_per_problem,avg_tokens_per_problem,min_tokens_per_problem,max_tokens_per_problem
0,4.83,1,20,4298.35,63,33098


## 3. Repair Usage / Recovery Statistics

정의:

- **problem-level repair 사용**: 해당 trajectory에서 `stage == "repair"`가 한 번 이상 등장
- **problem-level repair 복구 성공**: initial generate는 실패했고, 이후 repair stage에서 PASS가 발생한 문제
- **call-level repair 성공**: 개별 repair call의 status가 PASS


In [6]:
def compute_action_stats(step_df: pd.DataFrame, traj_df: pd.DataFrame, action_stage: str = "repair") -> dict:
    total_problems = len(traj_df)

    if step_df.empty or traj_df.empty:
        return {
            f"{action_stage}_used_problems": 0,
            f"{action_stage}_problem_usage_rate": 0.0,
            f"{action_stage}_recovered_problems": 0,
            f"{action_stage}_problem_recovery_rate": 0.0,
            f"{action_stage}_call_count": 0,
            f"{action_stage}_call_success_count": 0,
            f"{action_stage}_call_success_rate": 0.0,
        }

    stage_mask = step_df["stage"].eq(action_stage)
    used_tids = set(step_df.loc[stage_mask, "trajectory_id"].dropna())

    recovered_tids = set(
        step_df.loc[stage_mask & step_df["status"].eq("PASS"), "trajectory_id"].dropna()
    )

    used_problems = len(used_tids)
    recovered_problems = len(recovered_tids)

    call_count = int(stage_mask.sum())
    call_success_count = int((stage_mask & step_df["status"].eq("PASS")).sum())

    return {
        f"{action_stage}_used_problems": used_problems,
        f"{action_stage}_problem_usage_rate": used_problems / total_problems if total_problems else 0.0,
        f"{action_stage}_recovered_problems": recovered_problems,
        f"{action_stage}_problem_recovery_rate": recovered_problems / used_problems if used_problems else 0.0,
        f"{action_stage}_call_count": call_count,
        f"{action_stage}_call_success_count": call_success_count,
        f"{action_stage}_call_success_rate": call_success_count / call_count if call_count else 0.0,
    }

repair_stats = compute_action_stats(step_df, traj_df, "repair")
display(pd.DataFrame([repair_stats]).round(4))


,repair_used_problems,repair_problem_usage_rate,repair_recovered_problems,repair_problem_recovery_rate,repair_call_count,repair_call_success_count,repair_call_success_rate
0,57,0.3476,28,0.4912,628,28,0.0446


In [7]:
def print_action_stats(stats: dict, stage: str, total: int):
    used = stats[f"{stage}_used_problems"]
    recovered = stats[f"{stage}_recovered_problems"]
    calls = stats[f"{stage}_call_count"]
    call_success = stats[f"{stage}_call_success_count"]

    print("=" * 60)
    print(f"  [problem-level] {stage} 사용: {used}/{total} ({(used/total*100 if total else 0):.1f}%)")
    print(f"  [problem-level] {stage} 복구 성공: {recovered}/{used} ({(recovered/used*100 if used else 0):.1f}%)")
    print(f"  [call-level] {stage} 호출 누적: {calls}, 성공: {call_success}/{calls} ({(call_success/calls*100 if calls else 0):.1f}%)")
    print("=" * 60)

print_action_stats(repair_stats, "repair", len(traj_df))


  [problem-level] repair 사용: 57/164 (34.8%)
  [problem-level] repair 복구 성공: 28/57 (49.1%)
  [call-level] repair 호출 누적: 628, 성공: 28/628 (4.5%)


## 4. Plan / Replan Statistics

`rpe_policy` 또는 `policy_loop` 결과처럼 plan/replan stage가 있는 경우 자동 계산합니다.
repair-only 결과라면 0으로 표시됩니다.


In [8]:
for stage in ["plan", "plan_code", "replan"]:
    if "stage" in step_df and step_df["stage"].eq(stage).any():
        stats = compute_action_stats(step_df, traj_df, stage)
        display(pd.DataFrame([stats]).round(4))
        print_action_stats(stats, stage, len(traj_df))
    else:
        print(f"No stage found: {stage}")


No stage found: plan
No stage found: plan_code
No stage found: replan


## 5. Failure Breakdown

step-level / final trajectory-level에서 실패 유형을 확인합니다.


In [9]:
def coarse_status(status):
    if pd.isna(status):
        return "NONE"
    status = str(status)
    return "PASS" if status == "PASS" else status.split(":")[0]

if "status" in step_df:
    step_df["failure_family"] = step_df["status"].apply(coarse_status)
    display(step_df["status"].value_counts(dropna=False).rename("count").to_frame())
    display(step_df["failure_family"].value_counts(dropna=False).rename("count").to_frame())

if "final_status" in traj_df:
    traj_df["final_failure_family"] = traj_df["final_status"].apply(coarse_status)
    display(traj_df["final_status"].value_counts(dropna=False).rename("count").to_frame())
    display(traj_df["final_failure_family"].value_counts(dropna=False).rename("count").to_frame())


,count
status,
EXEC_FAIL:NameError,388
TEST_FAIL:AssertionError,155
PASS,135
EXEC_FAIL:SyntaxError,89
TEST_FAIL:IndexError,11
TEST_FAIL:NameError,4
EXEC_FAIL:AssertionError,3
TEST_FAIL:TypeError,3
EXEC_FAIL:TypeError,2


,count
failure_family,
EXEC_FAIL,484
TEST_FAIL,173
PASS,135


,count
final_status,
PASS,135
EXEC_FAIL:NameError,19
TEST_FAIL:AssertionError,4
EXEC_FAIL:SyntaxError,3
TEST_FAIL:TypeError,1
TEST_FAIL:NameError,1
TEST_FAIL:IndexError,1


,count
final_failure_family,
PASS,135
EXEC_FAIL,22
TEST_FAIL,7


## 6. Transition Analysis

각 trajectory의 `transition_path`를 사용해 coarse failure transition을 계산합니다.


In [10]:
from collections import Counter

transition_counter = Counter()

if "transition_path" in traj_df:
    for path in traj_df["transition_path"].dropna():
        if isinstance(path, str):
            # 혹시 문자열로 저장된 경우 대응
            try:
                path = json.loads(path)
            except Exception:
                path = [path]
        if not isinstance(path, list):
            continue
        for a, b in zip(path[:-1], path[1:]):
            ca = coarse_status(a)
            cb = coarse_status(b)
            transition_counter[f"{ca}->{cb}"] += 1

transition_df = (
    pd.DataFrame(transition_counter.items(), columns=["transition", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(transition_df)


,transition,count
0,EXEC_FAIL->EXEC_FAIL,407
1,TEST_FAIL->TEST_FAIL,109
2,TEST_FAIL->EXEC_FAIL,52
3,EXEC_FAIL->TEST_FAIL,32
4,EXEC_FAIL->PASS,23
5,TEST_FAIL->PASS,5


## 7. Recovered Problems

repair 또는 plan 계열 stage에서 PASS가 발생한 문제 목록을 확인합니다.


In [11]:
recovery_rows = []

for stage in sorted(step_df["stage"].dropna().unique()):
    if stage == "generate":
        continue
    hit = step_df[(step_df["stage"] == stage) & (step_df["status"] == "PASS")]
    for _, row in hit.iterrows():
        recovery_rows.append({
            "trajectory_id": row.get("trajectory_id"),
            "problem_id": row.get("problem_id"),
            "stage": stage,
            "call_index": row.get("call_index"),
            "input_tokens": row.get("input_tokens"),
            "output_tokens": row.get("output_tokens"),
            "total_tokens": row.get("total_tokens"),
        })

recovery_df = pd.DataFrame(recovery_rows)
display(recovery_df.sort_values(["stage", "call_index"]) if not recovery_df.empty else recovery_df)


,trajectory_id,problem_id,stage,call_index,input_tokens,output_tokens,total_tokens
2,HumanEval/43_run0,HumanEval/43,repair,1,583,45,628
3,HumanEval/46_run0,HumanEval/46,repair,1,851,512,1363
4,HumanEval/47_run0,HumanEval/47,repair,1,635,230,865
9,HumanEval/59_run0,HumanEval/59,repair,1,691,196,887
10,HumanEval/60_run0,HumanEval/60,repair,1,613,194,807
12,HumanEval/74_run0,HumanEval/74,repair,1,663,116,779
13,HumanEval/81_run0,HumanEval/81,repair,1,735,233,968
25,HumanEval/141_run0,HumanEval/141,repair,1,762,512,1274
26,HumanEval/147_run0,HumanEval/147,repair,1,497,103,600
0,HumanEval/40_run0,HumanEval/40,repair,2,702,83,785


## 8. Optional: Export Analysis Tables

분석 결과를 CSV로 저장합니다.


In [12]:
EXPORT_DIR = RESULT_DIR / "analysis_tables"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

stage_token_stats.reset_index().to_csv(EXPORT_DIR / "stage_token_stats.csv", index=False)
pd.DataFrame([repair_stats]).to_csv(EXPORT_DIR / "repair_stats.csv", index=False)

if not transition_df.empty:
    transition_df.to_csv(EXPORT_DIR / "transition_counts.csv", index=False)

if 'recovery_df' in globals() and not recovery_df.empty:
    recovery_df.to_csv(EXPORT_DIR / "recovered_problems.csv", index=False)

print("Saved analysis tables to:", EXPORT_DIR)


Saved analysis tables to: ../../results/phase2_qwen/humaneval/repair/analysis_tables
